# TechCompenso job scraping

Scrape job postings from techcompenso.com using Selenium.

## 1. Imports

In [14]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import sqlite3, json as _json, time

## 2. Launch browser and load jobs page

In [15]:
options = webdriver.ChromeOptions()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
driver = webdriver.Chrome(options=options)
driver.get("https://techcompenso.com/offerte-di-lavoro")
WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.TAG_NAME, "body")))
time.sleep(3)
print(driver.title)

Offerte di lavoro da remoto | TechBoard, smart working tech con RAL +40k€


## 3. Collect job cards

In [16]:
import re
cards = driver.find_elements(By.CSS_SELECTOR, "div[data-spotlight]")
print("num cards", len(cards))
jobs = []
badges = {"Hiring Partner"}
work_modes = {"Full-Remote", "Ibrido", "In sede", "Remote", "Hybrid"}
for card in cards:
    lines = [l.strip() for l in card.text.splitlines() if l.strip()]
    if lines and lines[0] in badges:
        lines = lines[1:]
    link = ""
    try:
        link = card.find_element(By.CSS_SELECTOR, "a[href*='/offerte-di-lavoro/']").get_attribute("href")
    except Exception:
        pass
    title = lines[0] if lines else None
    company = lines[1] if len(lines) > 1 else None
    salary = None
    work_mode = None
    location = None
    tags = []
    for ln in lines[2:]:
        if "€" in ln:
            salary = ln
        elif ln in work_modes:
            work_mode = ln
        elif location is None and not any(ch.isdigit() for ch in ln) and len(ln) < 40:
            location = ln
        else:
            tags.append(ln)
    salary_min = None
    salary_max = None
    if salary:
        nums = re.findall(r"[\d\.]+", salary)
        nums = [float(n.replace(".", "").replace(",", ".")) for n in nums]
        if "fino a" in salary.lower():
            if nums:
                salary_max = nums[0]
        else:
            if len(nums) >= 1:
                salary_min = nums[0]
            if len(nums) >= 2:
                salary_max = nums[1]
    jobs.append({"title": title, "company": company, "location": location, "work_mode": work_mode, "salary_min": salary_min, "salary_max": salary_max, "tags": ", ".join(tags), "url": link})
jobs = jobs[:15]
for j in jobs:
    print(j)
details = jobs

num cards 20
{'title': 'iOS Engineer', 'company': 'Welyk x Laura', 'location': 'Milano', 'work_mode': 'Ibrido', 'salary': "~ 50.000€ - 70.000€ all'anno", 'tags': '', 'url': 'https://techcompenso.com/offerte-di-lavoro/ios-engineer-welyk'}
{'title': 'Senior Frontend Developer', 'company': 'Welyk x Laura', 'location': 'Milano', 'work_mode': 'Ibrido', 'salary': "~ 50.000€ - 70.000€ all'anno", 'tags': '', 'url': 'https://techcompenso.com/offerte-di-lavoro/senior-frontend-developer-welyk'}
{'title': 'Android Developer', 'company': 'Welyk x Laura', 'location': 'Milano', 'work_mode': 'Ibrido', 'salary': "~ 50.000€ - 70.000€ all'anno", 'tags': '', 'url': 'https://techcompenso.com/offerte-di-lavoro/android-developer-welyk'}
{'title': 'Senior Backend Developer', 'company': 'Welyk x Laura', 'location': 'Milano', 'work_mode': 'Ibrido', 'salary': "~ 50.000€ - 70.000€ all'anno", 'tags': '', 'url': 'https://techcompenso.com/offerte-di-lavoro/senior-backend-developer-welyk'}
{'title': 'Backend Engineer

## 4. Persist to SQLite

In [17]:
db_path = "job_postings.db"
conn = sqlite3.connect(db_path)
df = pd.DataFrame(details)
if not df.empty:
    if "url" in df.columns:
        df = df.set_index("url")
    df.to_sql("techcompenso", conn, if_exists="replace", index=True, index_label="url")
    conn.commit()
    print(f"Saved {len(df)} postings")
conn.close()
driver.quit()

Saved 15 postings
